# Model Training

## Import necessary libraries

In [1]:
%pip install -qq -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Add current directory to Python path for imports
import os
import sys

# Add the parent directory (project root) to Python path so we can import from src
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
# Utility Functions
from src.utils import create_spark_session

# Create Spark session
spark, sedona = create_spark_session(app_name="ModelTrainingSpark")

## Loading Datasets

In [4]:
from src.utils import read_config_path

# Load data using configuration file
filepath = read_config_path(key="model_training_data_path", domain="processed")

df_raw = spark.read.csv(
    filepath,
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"',
    quote='"',
)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, StructType, StructField
from pyspark.ml.linalg import Vectors, VectorUDT
import re

schema = StructType(
    [
        StructField("timestamp_month", IntegerType(), True),
        StructField("timestamp_year", IntegerType(), True),
        StructField("resolution_time", IntegerType(), True),
        StructField("address_encoded", StringType(), True),
        StructField("latlong_encoded", StringType(), True),
        StructField("organization_encoded", StringType(), True),
        StructField("type_encoded", StringType(), True),
    ]
)

# ============================================================
# 2. UDF: Convert strings like "(2048,[834,1804],[1.0,1.0])"
#        into Spark SparseVector
# ============================================================


def parse_sparse(s):
    if s is None:
        return None
    # Example: (2048,[834,1804],[1.0,1.0])
    match = re.match(r"\((\d+),\[(.*?)\],\[(.*?)\]\)", s)
    if not match:
        return None

    size = int(match.group(1))

    # indices: convert "834,1804" → [834,1804]
    indices = match.group(2)
    indices = [int(x) for x in indices.split(",")] if indices else []

    # values: convert "1.0,1.0" → [1.0,1.0]
    values = match.group(3)
    values = [float(x) for x in values.split(",")] if values else []

    return Vectors.sparse(size, indices, values)


parse_sparse_udf = F.udf(parse_sparse, VectorUDT())

# ============================================================
# 3. UDF: Convert strings like "[13.67891,100.66709]"
#        into Spark DenseVector
# ============================================================


def parse_dense(s):
    if s is None:
        return None
    # Remove brackets → "13.67891,100.66709"
    s = s.strip()[1:-1]
    values = [float(x) for x in s.split(",")] if s else []
    return Vectors.dense(values)


parse_dense_udf = F.udf(parse_dense, VectorUDT())

# ============================================================
# 4. Apply conversions to each column
# ============================================================

df_prepared = (
    df_raw.withColumn("address_encoded", parse_sparse_udf("address_encoded"))
    .withColumn("organization_encoded", parse_sparse_udf("organization_encoded"))
    .withColumn("type_encoded", parse_sparse_udf("type_encoded"))
    .withColumn("latlong_encoded", parse_dense_udf("latlong_encoded"))
)

# ============================================================
# 5. SHOW FINAL RESULT
# ============================================================

df_prepared.show(5, truncate=False)

root
 |-- timestamp_month: integer (nullable = true)
 |-- timestamp_year: integer (nullable = true)
 |-- resolution_time: integer (nullable = true)
 |-- address_encoded: vector (nullable = true)
 |-- latlong_encoded: vector (nullable = true)
 |-- organization_encoded: vector (nullable = true)
 |-- type_encoded: vector (nullable = true)

+---------------+--------------+---------------+----------------------------+--------------------+-------------------------+--------------------+
|timestamp_month|timestamp_year|resolution_time|address_encoded             |latlong_encoded     |organization_encoded     |type_encoded        |
+---------------+--------------+---------------+----------------------------+--------------------+-------------------------+--------------------+
|9              |2021          |275            |(2048,[834,1804],[1.0,1.0]) |[13.67891,100.66709]|(1786,[10,53],[1.0,1.0]) |(25,[7,8],[1.0,1.0])|
|9              |2021          |253            |(2048,[348,426],[1.0,1.0])  |

In [6]:
df_prepared.printSchema()

root
 |-- timestamp_month: integer (nullable = true)
 |-- timestamp_year: integer (nullable = true)
 |-- resolution_time: integer (nullable = true)
 |-- address_encoded: vector (nullable = true)
 |-- latlong_encoded: vector (nullable = true)
 |-- organization_encoded: vector (nullable = true)
 |-- type_encoded: vector (nullable = true)



---

## Model Training

In [7]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressor, RandomForestRegressor
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator

# -----------------------------
# 1. Vector Assembler
# -----------------------------
assembler = VectorAssembler(
    inputCols=[
        "timestamp_month",
        "timestamp_year",
        "address_encoded",
        "latlong_encoded",
        "organization_encoded",
        "type_encoded",
    ],
    outputCol="features",
)

# -----------------------------
# 2. Regression Models
# -----------------------------
gbt = GBTRegressor(
    labelCol="resolution_time",
    featuresCol="features",
)

rf = RandomForestRegressor(
    labelCol="resolution_time",
    featuresCol="features",
)

# -----------------------------
# 3. Evaluators (Multi-Metric)
# -----------------------------
evaluators = {
    "rmse": RegressionEvaluator(
        labelCol="resolution_time", predictionCol="prediction", metricName="rmse"
    ),
    "mae": RegressionEvaluator(
        labelCol="resolution_time", predictionCol="prediction", metricName="mae"
    ),
    "r2": RegressionEvaluator(
        labelCol="resolution_time", predictionCol="prediction", metricName="r2"
    ),
}

# -----------------------------
# 4. Hyperparameter grids
# -----------------------------
gbt_paramGrid = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth, [3, 5])
    .addGrid(gbt.maxIter, [50, 100])
    .addGrid(gbt.stepSize, [0.05])
    .build()
)

rf_paramGrid = (
    ParamGridBuilder().addGrid(rf.numTrees, [100]).addGrid(rf.maxDepth, [8, 12]).build()
)

# -----------------------------
# 5. Pipelines
# -----------------------------
gbt_pipeline = Pipeline(stages=[assembler, gbt])
rf_pipeline = Pipeline(stages=[assembler, rf])

# -----------------------------
# 6. CrossValidators
# Use RMSE as main metric for tuning
# -----------------------------
cv_gbt = CrossValidator(
    estimator=gbt_pipeline,
    estimatorParamMaps=gbt_paramGrid,
    evaluator=evaluators["rmse"],
    numFolds=3,
    parallelism=1,
)

cv_rf = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=rf_paramGrid,
    evaluator=evaluators["rmse"],
    numFolds=3,
    parallelism=1,
)

In [ ]:
from pyspark.sql.functions import udf, col, concat_ws, desc
from pyspark.sql.types import ArrayType, IntegerType

# ============================================================
# 1. UDF to extract indices from SparseVector
# ============================================================


def get_indices(v):
    if v is None:
        return []
    return v.indices.tolist()


extract_indices_udf = udf(get_indices, ArrayType(IntegerType()))

# ============================================================
# 2. Add grouping columns
# ============================================================

df_grouped = (
    df_prepared.withColumn(
        "address_indices", extract_indices_udf(col("address_encoded"))
    )
    .withColumn("type_indices", extract_indices_udf(col("type_encoded")))
    .withColumn("address_group", concat_ws("_", col("address_indices")))
    .withColumn("type_group", concat_ws("_", col("type_indices")))
    .withColumn("strata", concat_ws("__", "address_group", "type_group"))
)

# ============================================================
# 3. Create sampling fractions (keep 10%)
# ============================================================

strata_values = [
    row["strata"] for row in df_grouped.select("strata").distinct().collect()
]
fractions = {s: 0.10 for s in strata_values}

# ============================================================
# 4. Stratified sampling
# ============================================================

sampled_df = df_grouped.sampleBy("strata", fractions, seed=42)

# ============================================================
# 5. Train-test split
# ============================================================

train_df, test_df = sampled_df.randomSplit([0.8, 0.2], seed=42)

# ============================================================
# 6. Check distribution (before vs after)
# ============================================================

before = df_grouped.groupBy("strata").count().withColumnRenamed("count", "before_count")

after = sampled_df.groupBy("strata").count().withColumnRenamed("count", "after_count")

distribution = (
    before.join(after, "strata", "left")
    .withColumn("sample_rate", col("after_count") / col("before_count"))
    .orderBy("before_count", ascending=False)
)

+-----------------------+------------+-----------+-----------+
|strata                 |before_count|after_count|sample_rate|
+-----------------------+------------+-----------+-----------+
|477_668__4_13_14       |1           |NULL       |NULL       |
|912_1681__2_3          |6           |NULL       |NULL       |
|1023_1534__2_14        |12          |NULL       |NULL       |
|1433_1804__0_6_9       |1           |NULL       |NULL       |
|742_1494__6_13_14_15_17|1           |NULL       |NULL       |
+-----------------------+------------+-----------+-----------+
only showing top 5 rows

+-----------------+------------+-----------+-----------+
|strata           |before_count|after_count|sample_rate|
+-----------------+------------+-----------+-----------+
|605_1758__3_7_18 |1           |1          |1.0        |
|794_912__0_4_5_6 |1           |1          |1.0        |
|2_102__4_7_16    |1           |1          |1.0        |
|7_292__7_18_21   |1           |1          |1.0        |
|1143_150

In [18]:
# Define Utility functions
import shutil
from pathlib import Path
from src.utils.ConfigUtils import get_data_dir


def evaluate_model(name, model):
    print(f"\n===== {name} Results =====")
    preds = model.transform(test_df)
    for metric, evaluator in evaluators.items():
        score = evaluator.evaluate(preds)
        print(f"{metric.upper()}: {score}")
    print("==========================")


def save_model(model, name: str | None = None):
    save_name = name or "model"

    # Base directory (without suffix)
    base_path = Path(get_data_dir()) / "model" / save_name

    # Old model directory
    old_path = Path(str(base_path) + "_cv_model_spark")

    # New temporary directory
    new_path = Path(str(base_path) + "_cv_model_spark_new")

    # 1. Remove the _new directory if it exists (cleanup)
    if new_path.exists():
        shutil.rmtree(new_path)

    # 2. Save model to the _new path
    model.save(str(new_path))

    # 3. Delete old model directory if exists
    if old_path.exists():
        shutil.rmtree(old_path)

    # 4. Rename _new → normal name
    new_path.rename(old_path)

    print(f"✔ Model saved successfully to: {old_path}")

In [11]:
# -----------------------------
# 8. Train both CV models
# -----------------------------
gbt_cv_model = cv_gbt.fit(train_df)

In [19]:
evaluate_model("GBTRegressor", gbt_cv_model)
save_model(gbt_cv_model, "gbt")


===== GBTRegressor Results =====
RMSE: 89.13410317462396
MAE: 52.28536478175962
R2: 0.5504812268162917
✔ Model saved successfully to: C:\Users\pun\Desktop\CU\CEDT-Y2-S1\2110403 - Introduction to Data Science and Data Engineering\CEDT-2110403-DSDE-Project\data\model\gbt_cv_model_spark


In [9]:
# -----------------------------
# 8. Train both CV models
# -----------------------------
rf_cv_model = cv_rf.fit(train_df)

In [10]:
rf_cv_model.save("rf_cv_model_spark")

In [21]:
evaluate_model("RandomForestRegressor", rf_cv_model)
save_model(rf_cv_model, "rf")


===== RandomForestRegressor Results =====
RMSE: 91.8582762602843
MAE: 54.32528705930575
R2: 0.5225843860885768
✔ Model saved successfully to: C:\Users\pun\Desktop\CU\CEDT-Y2-S1\2110403 - Introduction to Data Science and Data Engineering\CEDT-2110403-DSDE-Project\data\model\rf_cv_model_spark


In [ ]:
# spark.stop()

---